In [49]:
from ultralytics import YOLO
import cv2
import numpy as np
from util import get_parking_spots_bboxes, empty_or_not

In [50]:
yolo_model = YOLO("yolov8n.pt")

In [51]:
mask_path = './mask_1920_1080.png'
video_path = './data/parking_1920_1080.mp4'

In [52]:
mask = cv2.imread(mask_path, 0)

In [53]:
connected_components = cv2.connectedComponentsWithStats(mask, 4, cv2.CV_32S)
spots = get_parking_spots_bboxes(connected_components)

In [54]:
cap = cv2.VideoCapture(video_path)

In [55]:
frame_nmr = 0
spots_status = [False] * len(spots)
car_boxes = []

In [56]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 🔥 Resize frame (BIG speed boost)
    frame = cv2.resize(frame, (960, 540))

    # 🔵 Run YOLO every 5 frames
    if frame_nmr % 5 == 0:
        results = yolo_model(frame)[0]
        car_boxes = []

        for box in results.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            if cls in [2, 3, 5, 7] and conf > 0.4:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                car_boxes.append((x1, y1, x2, y2))

    # 🟢 Run SVM every 3 frames
    if frame_nmr % 3 == 0:
        for i, spot in enumerate(spots):
            x, y, w, h = spot

            # adjust coordinates after resize
            x = int(x * 960 / mask.shape[1])
            y = int(y * 540 / mask.shape[0])
            w = int(w * 960 / mask.shape[1])
            h = int(h * 540 / mask.shape[0])

            spot_crop = frame[y:y+h, x:x+w, :]

            if spot_crop.size != 0:
                spots_status[i] = empty_or_not(spot_crop)

    free_spots = 0

    # 🔴 Draw parking spots
    for i, spot in enumerate(spots):
        x, y, w, h = spot

        # adjust coordinates after resize
        x = int(x * 960 / mask.shape[1])
        y = int(y * 540 / mask.shape[0])
        w = int(w * 960 / mask.shape[1])
        h = int(h * 540 / mask.shape[0])

        if spots_status[i]:
            color = (0, 255, 0)
            free_spots += 1
        else:
            color = (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

    # 🔵 Draw YOLO detections
    for (x1, y1, x2, y2) in car_boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

    # Display count
    cv2.rectangle(frame, (50, 20), (300, 80), (0, 0, 0), -1)
    cv2.putText(frame, f'Free: {free_spots}/{len(spots)}',
                (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    cv2.imshow("YOLO + SVM Parking (Optimized)", frame)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    frame_nmr += 1

cap.release()
cv2.destroyAllWindows()


0: 384x640 (no detections), 91.8ms
Speed: 8.1ms preprocess, 91.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 94.0ms
Speed: 1.6ms preprocess, 94.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 82.1ms
Speed: 2.1ms preprocess, 82.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 84.0ms
Speed: 1.5ms preprocess, 84.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 tie, 83.2ms
Speed: 1.5ms preprocess, 83.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 tie, 91.7ms
Speed: 2.1ms preprocess, 91.7ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 tie, 90.8ms
Speed: 1.7ms preprocess, 90.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 106.1ms
Speed: 1.4ms preprocess, 106.1ms inference, 1.1ms postprocess 